# Notebook

### ENVELOPE Testing

In [1]:
import envelope_location as el
el.is_alive()

True

### Devices in Area List

In [68]:
#aoi = el.area(45.064924, 7.659707, 100)
aoi = el.area(45.060561941661376, 7.660129842020268, 100)
el.ipv4_addresses_in(aoi, max_age=1)

[{'publicAddress': '10.0.1.2', 'publicPort': 1234},
 {'publicAddress': '10.0.1.12', 'publicPort': 1234},
 {'publicAddress': '10.0.1.13', 'publicPort': 1234},
 {'publicAddress': '10.0.1.14', 'publicPort': 1234},
 {'publicAddress': '10.0.1.15', 'publicPort': 1234},
 {'publicAddress': '10.0.1.16', 'publicPort': 1234},
 {'publicAddress': '10.0.1.17', 'publicPort': 1234},
 {'publicAddress': '10.0.1.19', 'publicPort': 1234},
 {'publicAddress': '10.0.1.21', 'publicPort': 1234},
 {'publicAddress': '10.0.1.22', 'publicPort': 1234},
 {'publicAddress': '10.0.1.23', 'publicPort': 1234},
 {'publicAddress': '10.0.1.24', 'publicPort': 1234},
 {'publicAddress': '10.0.1.25', 'publicPort': 1234},
 {'publicAddress': '10.0.1.27', 'publicPort': 1234},
 {'publicAddress': '10.0.1.28', 'publicPort': 1234},
 {'publicAddress': '10.0.1.29', 'publicPort': 1234},
 {'publicAddress': '10.0.1.4', 'publicPort': 1234},
 {'publicAddress': '10.0.1.5', 'publicPort': 1234},
 {'publicAddress': '10.0.1.6', 'publicPort': 1234

### Subscription

In [ ]:
import json
from queue import Empty

# Use the host/IP that ENVELOPE can reach for the callback.
# NOTE: this must be THIS machine's reachable IP, not ENVELOPE's own IP (BASE).
CALLBACK_HOST = "192.168.0.128"
CALLBACK_PORT = 8081
CALLBACK_TIMEOUT_S = 60

events, sink, stop_receiver = el.start_receiver(
    port=CALLBACK_PORT,
    advertise_host=CALLBACK_HOST,
)
subscription_id = el.subscribe(aoi, sink, initial_event=True)

print(f"Subscription: {subscription_id}")
print(f"Callback sink: {sink}")

try:
    try:
        event = events.get(timeout=CALLBACK_TIMEOUT_S)
        print(json.dumps(event, indent=2))
    except Empty:
        print(f"No callback received within {CALLBACK_TIMEOUT_S} seconds")
finally:
    el.unsubscribe(subscription_id)
    stop_receiver()
    print(f"Unsubscribed: {subscription_id}")

### IONUT Testing

In [ ]:
import os
import uuid
from datetime import datetime, timezone

import escalator as dc

In [15]:
# Configure these values for the target service.
RISK_SERVICE_BASE_URL = "http://192.168.0.174:30090"
EVALUATOR_TOKEN = os.environ.get("EVALUATOR_BEARER_TOKEN", "devsecret_replace_in_deployment_evaluator_token")

endpoint = RISK_SERVICE_BASE_URL.rstrip("/") + "/api/v1/risk-events"
event_id = str(uuid.uuid4())

body = {
    "eventId": event_id,
    "riskLevel": 4,
    "occurredAt": datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z"),
    "devices": [
        {
            "ipv4Address": {
                "publicAddress": "8.8.8.8",
                "privateAddress": "192.168.1.10"
            }
        }
    ],
    "aoiId": "turin-masa-aoi-1",
}

body

{'eventId': '01f6964d-a1ce-4b7f-946c-88b95f7d6de2',
 'riskLevel': 4,
 'occurredAt': '2026-09-18T12:30:39.372Z',
 'devices': [{'ipv4Address': {'publicAddress': '8.8.8.8',
    'privateAddress': '192.168.1.10'}}],
 'aoiId': 'turin-masa-aoi-1'}

In [16]:
success = dc.post_risk_event(
    url=endpoint,
    token=EVALUATOR_TOKEN,
    event_id=event_id,
    body=body,
    timeout=10,
    retries=1,
    backoff_s=1,
)

print("Accepted:", success)

Accepted: True
